## LDA LaBraM performance comparison

In [4]:
import os
from os import mkdir
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from core.datasets import KFoldSplitter, ClassFinetuneDataset
from core import settings, plot_data_processing
from core import io
from sklearn.linear_model import LogisticRegression

from core.io_models import Trial

In [5]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir('..')

labram_model_name = "class_nomask_10_22"
run_model_train = False

labram_model_dir = settings.FINETUNE_OUTPUT_DIR / labram_model_name
labram_inference_dir = settings.INFERENCE_DIR / labram_model_name
k_folds = 6
target_mode = "classify"  # the splitters load the input epochs only

# the same trials, window and labels as the LaBraM run: its baseline folds from keys.json, and every fold's post-patching
# test trials from the postpatch_keys.json its inference saved
if run_model_train:
    splitter = KFoldSplitter(input_data_dir=settings.FINETUNE_INPUT_EPOCHS_DIR, target_data_dir=None,
                             target_mode=target_mode, seed=settings.SEED, k_folds=k_folds,
                             no_press_threshold=settings.NO_PRESS_FROM_SUBJECT)

    folds = splitter.split_from_keys(io.load_training_split_keys(labram_model_dir, k_folds))

    postpatch_splitter = KFoldSplitter(input_data_dir=settings.PATCH_INPUT_EPOCHS_DIR, target_data_dir=None,
                                       target_mode=target_mode, seed=settings.SEED, k_folds=k_folds,
                                       no_press_threshold=settings.NO_PRESS_FROM_SUBJECT)
    postpatch_fold_keys = io.load_training_split_keys(labram_inference_dir, k_folds, filename="postpatch_keys.json")

    # interpolated channels of every trial, recorded in metadata["allowed"] (the non-masked LaBraM run saw them as normal inputs)
    interpolations_baseline = io.load_interpolated_channels(settings.FINETUNE_DIR / "epochs_manifest.csv")
    interpolations_postpatch = io.load_interpolated_channels(settings.PATCH_DIR / "epochs_manifest.csv")
    class_window = {"epoch_window": settings.EPOCH_WINDOW, "patch_window": settings.CLASS_PATCH_WINDOW, "fs": settings.RSFREQ,
                    "condition_classes": settings.CONDITION_CLASS_IDS}

    output_dir = settings.INFERENCE_DIR / "logistic_regression"
    for i, fold in enumerate(folds):

        train, val, test = fold.initialize_class_datasets(**class_window, interpolations=interpolations_baseline)
        postpatch_dataset = ClassFinetuneDataset(epochs_dict=postpatch_splitter.select(postpatch_fold_keys[i]["test"]),
                                                 **class_window, interpolations=interpolations_postpatch)

        X, y, meta = train.get_stacked_items()
        X_val, y_val, meta_val = val.get_stacked_items()
        X_test, y_test, meta_test = test.get_stacked_items()
        X_postpatch, y_postpatch, meta_postpatch = postpatch_dataset.get_stacked_items()

        # choose C based on validation
        best_score, best_model = 0, None
        for c in np.logspace(-4, 2, num=6):
            model = make_pipeline(StandardScaler(), LogisticRegression(penalty="l2", solver="lbfgs", C=c, max_iter=2000, random_state=settings.SEED))
            model.fit(X, y)
            bal_acc = balanced_accuracy_score(y_val, model.predict(X_val))
            if bal_acc > best_score:
                best_score = bal_acc
                best_model = model
            print(f"Fold {i}, C={c:.4f}: Train R2={model.score(X, y):.4f}, Val R2={bal_acc:.4f}, Best R2={best_score:.4f}")

        test_logits = best_model.decision_function(X_test) # n trials x 8 (logits)
        postpatch_logits = best_model.decision_function(X_postpatch)

        print(f"Fold {i + 1}: test balanced accuracy {balanced_accuracy_score(y_test, test_logits.argmax(axis=1)):.4f}")

        # one file per session in the layout of run_inference.py: per trial the class id, the class logits as one row (1, n_classes),
        # so the mean over the first axis gives the class logits as for LaBraM's per-channel output, and the interpolated channels
        fold_dir = output_dir / str(i + 1)
        fold_dir.mkdir(parents=True, exist_ok=True)
        for session, logits, labels, metas in [("baseline", test_logits, y_test, meta_test), ("postpatch", postpatch_logits, y_postpatch, meta_postpatch)]:
            with h5py.File(fold_dir / f"{session}_reconstruction.h5", "w") as h5file:
                for trial_logits, label, trial_meta in zip(logits, labels, metas):
                    subj, cond, idx = trial_meta["subject"], trial_meta["condition"], trial_meta["id"]
                    io.save_inference_trial_hdf(subj, cond, idx, h5file, int(label), trial_logits[None, :], window_metrics={}, patch=None)
                    h5file[f"{subj}/{cond}/{idx}"].attrs["interpolated"] = np.flatnonzero(~trial_meta["allowed"].numpy())

In [6]:
# load the saved data
def load_filtered_merged(model_name):
    baseline_merged = []
    postpatch_merged = []

    for i in range(1, 7):
        b_file_path = settings.INFERENCE_DIR / model_name / str(i) / "baseline_reconstruction.h5"
        p_file_path = settings.INFERENCE_DIR / model_name / str(i) / "postpatch_reconstruction.h5"

        b_fold_trials = io.load_inference_trials(b_file_path, subjects=None, conditions=None, idxs=None, exclude_subjects=settings.EXCLUDED_SUBJECTS, window="full", stim_patch=None)
        baseline_merged.extend(b_fold_trials)
        p_fold_trials = io.load_inference_trials(p_file_path, subjects=None, conditions=None, idxs=None, exclude_subjects=settings.EXCLUDED_SUBJECTS, window="full", stim_patch=None)
        postpatch_merged.extend(p_fold_trials)
    b_df = plot_data_processing.class_trials_frame(baseline_merged, model_name, session="baseline")
    p_df = plot_data_processing.class_trials_frame(postpatch_merged, model_name, session="postpatch")
    return pd.concat([b_df, p_df], ignore_index=True)

logreg_df = load_filtered_merged("logistic_regression", )
labram_df = load_filtered_merged("class_nomask_10_22")
labram_df["model"] = "labram"

compare_df = pd.concat([logreg_df, labram_df], ignore_index=True)

Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading inference trials...
Filtering for subjects: all
Filtering for conditions: all
Filtering for idxs: all
Loading in

In [7]:
def participant_scores(df: pd.DataFrame):
    proba = df[[f"p{k}" for k in range(8)]].to_numpy()
    auc = roc_auc_score(df["label"], proba, multi_class="ovr", average=None, labels=range(8))
    bal_acc = balanced_accuracy_score(df["label"], df["pred"])
    auc_format = {f"auc_{k}": a for k, a in enumerate(auc)}

    return pd.Series({"bal_acc": bal_acc, **auc_format, "n_trials": len(df)})


per_subj_scores = compare_df.groupby(["model", "session", "subject"]).apply(participant_scores).reset_index()

display(per_subj_scores.groupby(["model", "session"]).agg({"bal_acc": ["mean", "sem", "count"]}))


bal_acc                
                                   mean       sem count
model               session                            
labram              baseline   0.220115  0.005398    55
                    postpatch  0.216091  0.006395    55
logistic_regression baseline   0.235907  0.005668    55
                    postpatch  0.226855  0.006492    55